In [ ]:
import os

# policy
from huggingface_hub import snapshot_download
from lerobot.configs.policies import PreTrainedConfig

# lerobot
from lerobot.record import record, RecordConfig, DatasetRecordConfig
from lerobot.constants import OBS_ENV_STATE 

# torch
from torch import cuda

# gemini
from google import genai

# utils
import pprint
from dotenv import load_dotenv

# my code
from robot.robot_config import robot_config
from robot.robot_const import FOLDED_START_POSE
from src.paths import REPO_ROOT, HF_NAME, POLICIES_DIR, EVAL_DIR
from src.utils import check_resume
from gemini.gemini_lerobot_processor import GeminiAnnotateProcessorStep
from gemini.gemini_prompts import YELLOW_CAR_PROMPT


# set up env secrets
load_dotenv(REPO_ROOT / ".env", override=True)

# cuda
device = "cuda" if cuda.is_available() else "cpu"
print(f"Using device: {device}")

%load_ext autoreload
%autoreload 2

/home/jonathan/miniforge3/envs/lerobot-sim-real-sim/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


#### 1. Set Evaluation Params:

In [2]:
PUSH_TO_HUB       = False
SAVE_TO_DATASET   = False
REPO_NAME         = 'so101_car_pick_and_place'
EXPERIMENT_NAME   = 'extended_v2'
POLICY_CHECKPOINT = '060000'
POLICY_TYPE       = 'act'
NUM_EPISODES      = 1
FPS               = 30
EPISODE_TIME_SEC  = 600
RESET_TIME_SEC    = 5
EVAL_TYPE         = '12_test_case'

Log to HF

In [ ]:
if PUSH_TO_HUB:
    os.system(f"hf auth login --token {os.getenv('HUGGINGFACE_TOKEN')}")

#### 2. Initialize Policy and Overrides:

In [4]:
assert len(POLICY_CHECKPOINT) == 6

# Resolve path or HF id
if POLICY_CHECKPOINT is None:
    POLICY_CHECKPOINT = "last"
policy_path = POLICIES_DIR / POLICY_TYPE / REPO_NAME / EXPERIMENT_NAME / "checkpoints" / POLICY_CHECKPOINT / "pretrained_model"

if not policy_path.exists():
    print('Loading policy from HF')
    policy_id = f"{HF_NAME}/{POLICY_TYPE}-{REPO_NAME}-{EXPERIMENT_NAME}"
    snapshot_download(
    repo_id                = policy_id,
    revision               = "main",
    local_dir              = str(policy_path),
    local_dir_use_symlinks = False
    )

policy_config = PreTrainedConfig.from_pretrained(policy_path)
policy_config.pretrained_path = policy_path # type: ignore
pprint.pprint(policy_config.input_features)

{'observation.environment_state': PolicyFeature(type=<FeatureType.ENV: 'ENV'>,
                                                shape=(3,)),
 'observation.images.top_cam': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>,
                                             shape=(3, 480, 640)),
 'observation.images.wrist_cam': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>,
                                               shape=(3, 480, 640)),
 'observation.state': PolicyFeature(type=<FeatureType.STATE: 'STATE'>,
                                    shape=(6,))}


Policy Overrides:

In [5]:
print(f"Original n_action_steps = {policy_config.n_action_steps}, temporal_ensemble = {policy_config.temporal_ensemble_coeff}")
# policy_config.n_action_steps = 100
# policy_config.temporal_ensemble_coeff = 0.01
print(f"Override n_action_steps = {policy_config.n_action_steps}, temporal_ensemble = {policy_config.temporal_ensemble_coeff}")

Original n_action_steps = 100, temporal_ensemble = None
Override n_action_steps = 100, temporal_ensemble = None


#### 3. Specify additional feature for Gemini annotation

In [6]:
new_feature_name = OBS_ENV_STATE
new_feature_meta = {
    "dtype": "float32",
    "names": [
        "x_px",
        "y_px",
        "rotation_deg"
    ],
    "shape":(3,)
}
new_feature_list = [{new_feature_name: new_feature_meta}]

And build the preprocessor that will supply this data during inference:

In [ ]:
client = genai.Client(api_key=os.getenv('GOOGLE_API_KEY'))
gemini_processor = GeminiAnnotateProcessorStep(client = client, prompt = YELLOW_CAR_PROMPT)

#### 4. Build Evaluation Dataset
Used for data storage of the evaluation dataset

In [8]:
dataset_path = EVAL_DIR / POLICY_TYPE / REPO_NAME / f"{EXPERIMENT_NAME}-{EVAL_TYPE}"
resume = check_resume(dataset_path)

dscfg = DatasetRecordConfig(
    repo_id                             = f"{HF_NAME}/eval_{REPO_NAME}_{EXPERIMENT_NAME}_{EVAL_TYPE}",
    single_task                         = f"eval dataset for {REPO_NAME} with policy {EXPERIMENT_NAME}, mode = {EVAL_TYPE}",
    root                                = dataset_path.__str__(),
    fps                                 = FPS,
    episode_time_s                      = EPISODE_TIME_SEC,
    reset_time_s                        = RESET_TIME_SEC,
    num_episodes                        = NUM_EPISODES,
    video                               = True,
    push_to_hub                         = PUSH_TO_HUB,
    private                             = True,
    num_image_writer_processes          = 0,
    num_image_writer_threads_per_camera = 4,
    video_encoding_batch_size           = 1,
)

rc = RecordConfig(
    robot        = robot_config,
    dataset      = dscfg,
    teleop       = None,
    policy       = policy_config,
    display_data = True,
    play_sounds  = True,
    resume       = resume
)

#### 5. Run inference while calling Gemini for environemnt evaluation

In [ ]:
check_resume(dataset_path)
record(rc, 
    extra_robot_processor = [gemini_processor],
    extra_features        = new_feature_list,
    save_to_ds            = SAVE_TO_DATASET,
    reset_pose            = FOLDED_START_POSE,
    give_feedback         = False,
    log_to_file           = False)

INFO 2025-10-11 21:02:57 t/record.py:431 {'dataset': {'episode_time_s': 600,
             'fps': 30,
             'num_episodes': 1,
             'num_image_writer_processes': 0,
             'num_image_writer_threads_per_camera': 4,
             'private': True,
             'push_to_hub': False,
             'rename_map': {},
             'repo_id': 'jonathm126/eval_so101_car_pick_and_place_extended_v2_12_test_case',
             'reset_time_s': 5,
             'root': '/home/jonathan/Github/IDC_Project-Sim_Real_Sim/eval/act/so101_car_pick_and_place/extended_v2-12_test_case',
             'single_task': 'eval dataset for so101_car_pick_and_place with '
                            'policy extended_v2, mode = 12_test_case',
             'tags': None,
             'video': True,
             'video_encoding_batch_size': 1},
 'display_data': True,
 'play_sounds': True,
 'policy': {'chunk_size': 200,
            'device': 'cuda',
            'dim_feedforward': 3200,
            'dim_model

Loading weights from local directory


INFO 2025-10-11 21:03:00 a_opencv.py:180 OpenCVCamera(/dev/v4l/by-id/usb-Sonix_Technology_Co.__Ltd._USB2.0_CAM1_USB2.0_CAM1-video-index0) connected.
INFO 2025-10-11 21:03:02 a_opencv.py:180 OpenCVCamera(/dev/v4l/by-id/usb-046d_Logitech_BRIO_8F54E371-video-index0) connected.
INFO 2025-10-11 21:03:02 follower.py:104 so_101_follower SO101Follower connected.
INFO 2025-10-11 21:03:03 ls/utils.py:227 Recording episode 0
[2025-10-11T18:03:03Z INFO  winit::platform_impl::linux::x11::window] Guessed window scale factor: 1
[2025-10-11T18:03:03Z WARN  wgpu_hal::gles::egl] No config found!
[2025-10-11T18:03:03Z WARN  wgpu_hal::gles::egl] EGL says it can present to the window but not natively
[2025-10-11T18:03:03Z WARN  wgpu_hal::gles::adapter] Max vertex attribute stride unknown. Assuming it is 2048
[2025-10-11T18:03:03Z WARN  wgpu_hal::vulkan::conv] Unrecognized present mode 1000361000
[2025-10-11T18:03:03Z WARN  wgpu_hal::gles::adapter] Max vertex attribute stride unknown. Assuming it is 2048


Value(False)


[2025-10-11T18:03:03Z WARN  wgpu_hal::vulkan::conv] Unrecognized present mode 1000361000
[2025-10-11T18:03:03Z WARN  wgpu_hal::vulkan::conv] Unrecognized present mode 1000361000
[2025-10-11T18:03:03Z WARN  wgpu_hal::vulkan::conv] Unrecognized present mode 1000361000
[2025-10-11T18:03:04Z WARN  wgpu_hal::vulkan::conv] Unrecognized present mode 1000361000
[2025-10-11T18:03:04Z WARN  wgpu_hal::vulkan::conv] Unrecognized present mode 1000361000


Value(False)
Left arrow key pressed. Exiting loop and rerecord the last episode...
Right arrow key pressed. Exiting loop...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Left arrow key pressed. Exiting loop and rerecord the last episode...
